### Experimental vs simulated P(s)

Compares contact-probability-versus-genomic-separation curves between scsHi-C
and simulation data, for both *cis* (within a
chromatid) and *trans* (between the two sister chromatids).

Simulated inputs are the `cis_matrix.h5` / `trans_matrix.h5` files written by
`contacts_vs_distance_from_trajectories.py`, which also provides the P(s)
helpers used here.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from contacts_vs_distance_from_trajectories import (
    load_exp_ps,
    load_sim_ps,
    norm_factor,
    trajectory_dir,
)

#### Configuration

In [ ]:
# --- experimental data (Mitter et al. 2020, GSE152373) ---
EXP_ROOT = "../experimental_data"
EXP_CIS_COOL = f"{EXP_ROOT}/GSM4613674_G2.fc_1_2_3_4.wOldG2.cis.1000.cool"
EXP_TRANS_COOL = f"{EXP_ROOT}/GSM4613674_G2.fc_1_2_3_4.wOldG2.trans.1000.cool"

# Both experiment and simulation are normalised to their own cis P(s) at this
# separation, so the curves can be overlaid on one scale.
NORM_DIST_BP = 4e4

NPROC = 16

# Parameter points to overlay. Add a row to plot another alpha.
SIM_RUNS = [
    # (alpha, tau, condition, colour)
    (50000, 11, "WT12h", "#24a645"),
]

#### Load experimental P(s)

In [ ]:
exp_cis_dist, exp_cis_ps = load_exp_ps(EXP_CIS_COOL, nproc=NPROC)
exp_trans_dist, exp_trans_ps = load_exp_ps(EXP_TRANS_COOL, nproc=NPROC)

exp_norm = norm_factor(exp_cis_dist, exp_cis_ps, NORM_DIST_BP)
print(f"experimental cis norm factor at {NORM_DIST_BP:.0e} bp: {exp_norm:.6g}")

#### Load simulated P(s)

In [ ]:
sim_curves = []

for alpha, tau, condition, colour in SIM_RUNS:
    path_name = trajectory_dir(condition, alpha, tau)

    cis_dist, cis_ps = load_sim_ps(path_name, "cis", nproc=NPROC)
    trans_dist, trans_ps = load_sim_ps(path_name, "trans", nproc=NPROC)

    # trans is divided by the *cis* norm factor on purpose, so that the
    # cis/trans ratio is preserved on the plot
    norm = norm_factor(cis_dist, cis_ps, NORM_DIST_BP)

    sim_curves.append({
        "alpha": alpha,
        "tau": tau,
        "condition": condition,
        "colour": colour,
        "cis_dist": cis_dist,
        "cis_ps": cis_ps,
        "trans_dist": trans_dist,
        "trans_ps": trans_ps,
        "norm": norm,
    })

    print(f"alpha={alpha}, tau={tau}: cis norm factor {norm:.6g}")

#### Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

for run in sim_curves:
    alpha = run["alpha"]
    label = rf"Simu cis, $\alpha$={alpha}"

    ax.loglog(run["cis_dist"], run["cis_ps"] / run["norm"],
              color=run["colour"], linewidth=2, label=label)
    ax.loglog(run["trans_dist"], run["trans_ps"] / run["norm"],
              color=run["colour"], linewidth=1.5, linestyle="--")

ax.loglog(exp_cis_dist, exp_cis_ps / exp_norm,
          color="#A04294", linewidth=1.5, label="Exp cis")
ax.loglog(exp_trans_dist, exp_trans_ps / exp_norm,
          color="#A04294", linewidth=1.5, linestyle="--", label="Exp trans")

ax.set_xlim(4e4, 1e8)
ax.set_ylim(6e-5, 0.15e1)
ax.set_aspect("equal")

ax.tick_params(axis="both", which="major", length=9, width=1.5)
ax.tick_params(axis="both", which="minor", length=4, width=0.75)

ax.set_xlabel("Genomic distance [bp]", fontsize=26)
ax.set_ylabel("Contact frequency", fontsize=26)
ax.tick_params(labelsize=24)
ax.set_title("WT", fontsize=20)
ax.grid(True)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=14)

plt.show()